# Wprowadzenie do lokalnych LLM z Ollamą​ 🦙

# 🐍 Notanik 2 — Ollama + Python

W tym notatniku będziemy komunikować się z lokalnym modelem Ollama **bezpośrednio przez bibliotekę `ollama`**

## Wymagania

Upewnij się, że masz Zainstalowaną bibliotekę:

```bash
pip install ollama
```

Zacznijmy od zaimportowania biblioteki i sprawdzenia dostępnych modeli:


In [8]:
import ollama

# Sprawdzenie połączenia — lista dostępnych modeli
models = ollama.list()
print("✅ Dostępne modele:")
for m in models['models']:
    print(f"  - {m["model"]}")

✅ Dostępne modele:
  - granite4.1:3b
  - qwen2.5:0.5b
  - granite3.1-moe:1b
  - nemotron-mini:latest
  - gemma3:1b
  - gpt-oss:20b
  - gemma4:31b
  - gemma3:12b
  - qwen3.5:4b
  - qwen3.5:9b
  - gemma3:4b
  - gemma4:26b
  - gemma4:e2b
  - gemma4:e4b


---
## 🟢 Zadanie 1 — Generator JSON

Poprosimy model o wygenerowanie danych w formacie JSON, a następnie przetworzymy je w Pythonie.

Cel: Nauczyenie się, jak zmusić model AI do zwracania odpowiedzi w uporządkowanym formacie oraz jak później odczytać te dane w kodzie.

W świecie LLM bardzo często nie chcesz dostawać „luźnego tekstu”, tylko dane w konkretnej strukturze, np. JSON. Dzięki temu aplikacja może łatwo zrozumieć odpowiedź modelu i automatycznie ją wykorzystać.
>
> 💬 **Prompt:** Wygeneruj listę 3 kursów online o Pythonie w formacie JSON.
> Każdy kurs powinien mieć pola: `nazwa`, `poziom`, `czas_godziny`.
> Odpowiedz TYLKO czystym JSON, bez żadnych komentarzy.

In [9]:
import json

MODEL = "gemma4:e4b"  

response = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": (
                "Wygeneruj listę 3 kursów online o Pythonie w formacie JSON. "
                "Każdy kurs powinien mieć pola: nazwa, poziom, czas_godziny. "
                "Odpowiedz TYLKO czystym JSON, bez żadnych komentarzy."
            )
        }
    ]
)

raw = response['message']['content']
print("📄 Surowa odpowiedź modelu:")
print(raw)

📄 Surowa odpowiedź modelu:
```json
[
  {
    "nazwa": "Wprowadzenie do Pythona dla początkujących",
    "poziom": "początkujący",
    "czas_godziny": 15
  },
  {
    "nazwa": "Programowanie obiektowe w Pythonie",
    "poziom": "średniozaawansowany",
    "czas_godziny": 25
  },
  {
    "nazwa": "Zaawansowane algorytmy i struktury danych z Pythonem",
    "poziom": "zaawansowany",
    "czas_godziny": 35
  }
]
```


Czasem zdarza sie, że odpowiedz nie zawiera śmieci dlatego przed kolejnym krokiem je wyczyścimy oraz zrobimy obiekt do dalszego uzytku:

In [10]:

clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()

kursy = json.loads(clean)

print("Kursy:")
for kurs in kursy:
    print(f"  [{kurs['poziom']}] {kurs['nazwa']} — {kurs['czas_godziny']}h")

📚 Kursy:
  [początkujący] Wprowadzenie do Pythona dla początkujących — 15h
  [średniozaawansowany] Programowanie obiektowe w Pythonie — 25h
  [zaawansowany] Zaawansowane algorytmy i struktury danych z Pythonem — 35h


### ✏️ Twoja kolej!

Zmodyfikuj prompt, aby model wygenerował listę **5 pracowników działu IT** z polami:
`imie`, `stanowisko`, `lata_doswiadczenia`.

Następnie wyfiltruj i wypisz tylko tych z doświadczeniem > 3 lata.

In [11]:
# TODO: Twój kod tutaj


---
## 🔵 Zadanie 2 — Chatbot z pamięcią rozmowy

W tym zadaniu stworzymy prostego chatbota, który pamięta wcześniejsze wiadomości i potrafi prowadzić ciągłą rozmowę.

Ważne: model AI sam z siebie nie pamięta poprzednich pytań ani odpowiedzi. Przy każdym nowym zapytaniu trzeba ponownie przekazać historię rozmowy, aby model zachował kontekst.
>
> ▶️ Uruchom komórkę i rozmawiaj w terminalu. Wpisz `quit` aby zakończyć.

In [15]:
historia = [
    {
        "role": "system",
        "content": "Jesteś pomocnym asystentem. Odpowiadaj po polsku, zwięźle i rzeczowo."
    }
]

print("🤖 Chatbot uruchomiony. Wpisz 'quit' aby zakończyć.\n")

while True:
    user_input = input("Ty: ").strip()
    if user_input.lower() in ["quit", "exit", "wyjdź"]:
        print("👋 Do widzenia!")
        break
    if not user_input:
        continue

    historia.append({"role": "user", "content": user_input})

    response = ollama.chat(model=MODEL, messages=historia)
    odpowiedz = response['message']['content']

    historia.append({"role": "assistant", "content": odpowiedz})

    print(f"\n🤖 AI: {odpowiedz}\n")

print(f"\n📊 Długość historii: {len(historia)} wiadomości")

🤖 Chatbot uruchomiony. Wpisz 'quit' aby zakończyć.


🤖 AI: Dobrze, dziękuję. Jestem gotów ci pomóc.


🤖 AI: Jestem modelem językowym, więc nie mam wieku w tradycyjnym rozumieniu. Nie starzeję się.

👋 Do widzenia!

📊 Długość historii: 5 wiadomości


### 💡 Obserwacja

Sprawdź, czy model pamięta wcześniejsze wiadomości. Przykładowe pytania do testu:
1. *"Jak mam na imię? Nazywam się Anna."*
2. *(po kilku zdaniach)* *"Przypomnij mi jak mam na imię"*

---
## 🔴 Zadanie 3 — Streszczanie pliku TXT

Wczytamy plik `artykul_llm.txt` wygenerowany w Warsztacie 1 i poprosimy model o:
1. 📝 Streszczenie w 3 zdaniach
2. 🔑 Wyciągnięcie kluczowych punktów
3. 💾 Zapis wyników do pliku `streszczenie.txt`

> ⚠️ Upewnij się, że plik `artykul_llm.txt` znajduje się w tym samym katalogu!

In [ ]:
# Wczytanie pliku
try:
    with open("artykul_llm.txt", "r", encoding="utf-8") as f:
        tekst = f.read()
    print(f"✅ Wczytano plik ({len(tekst)} znaków)")
    print("\n--- Podgląd (pierwsze 300 znaków) ---")
    print(tekst[:300] + "...")
except FileNotFoundError:
    print("❌ Nie znaleziono pliku artykul_llm.txt")
    print("   Wróć do Warsztatu 1 i wykonaj Zadanie 6!")
    tekst = None

In [ ]:
# Streszczenie

if tekst:
    resp_streszczenie = ollama.chat(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": f"Streść poniższy tekst w dokładnie 3 zdaniach po polsku:\n\n{tekst}"
        }]
    )
    streszczenie = resp_streszczenie['message']['content']

    print("📝 Streszczenie:")
    print(streszczenie)

In [ ]:
# Kluczowe punkty

if tekst:
    resp_punkty = ollama.chat(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": (
                f"Na podstawie poniższego tekstu wypisz 5 najważniejszych punktów "
                f"jako lista numerowana po polsku:\n\n{tekst}"
            )
        }]
    )
    punkty = resp_punkty['message']['content']

    print("🔑 Kluczowe punkty:")
    print(punkty)

In [ ]:
if tekst:
    # Zapis do pliku
    wynik = f"=== STRESZCZENIE ===\n\n{streszczenie}\n\n"
    wynik += f"=== KLUCZOWE PUNKTY ===\n\n{punkty}\n"

    with open("streszczenie.txt", "w", encoding="utf-8") as f:
        f.write(wynik)

    print("✅ Wyniki zapisane do streszczenie.txt")
    print(f"   Rozmiar pliku: {len(wynik)} znaków")

### ✏️ Twoja kolej!

Rozbuduj analizę pliku — dodaj trzecie zapytanie do modelu, które:
- oceni **sentyment** tekstu (pozytywny / neutralny / negatywny)
- zaproponuje **3 pytania** do dyskusji na temat artykułu
`

In [ ]:
# TODO: Twój kod tutaj


---
## 🌟 Bonus — Streaming (odpowiedź na żywo)

Zamiast czekać na całą odpowiedź, możemy wyświetlać ją **token po tokenie** — tak jak na ChatGPT.

In [ ]:
import sys

print("🌊 Streaming odpowiedź:\n")

stream = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "Wyjaśnij czym jest sieć neuronowa w 4 zdaniach."}],
    stream=True
)

for chunk in stream:
    token = chunk['message']['content']
    print(token, end="", flush=True)

print("\n\n✅ Koniec streamingu")